# Image Captioning Deep Learning Fine-Tuning (Google Colab GPU)

This notebook fine-tunes **Salesforce BLIP** on the **Flickr30k** dataset using a free Cloud GPU (T4/V100).

### Why train in Google Colab instead of a local laptop?
- Training locally on an 8GB CPU takes hours and can exhaust RAM.
- Google Colab provides a free 15GB T4 GPU which trains in **10-15 minutes** with full mixed-precision (`fp16`).
- After training, the model weights are automatically zipped for easy download and drop-in use in your local project.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Install dependencies
!pip install -q transformers==4.49.0 datasets==3.6.0 accelerate torchvision pillow

In [ ]:
import os
import torch
from datasets import load_dataset
from PIL import Image
from transformers import (
    BlipForConditionalGeneration,
    BlipProcessor,
    Trainer,
    TrainingArguments,
)

BASE_MODEL = "Salesforce/blip-image-captioning-base"
OUTPUT_DIR = "./blip-flickr30k"

print("PyTorch version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained(BASE_MODEL)
model = BlipForConditionalGeneration.from_pretrained(BASE_MODEL).to(device)

# Optional: Freeze vision model for faster convergence
for param in model.vision_model.parameters():
    param.requires_grad = False

print("Model loaded on", device)

In [ ]:
# 3. Load Flickr30k Dataset
dataset = load_dataset("nlphuji/flickr30k", split="train")

# Train/validation split
split = dataset.train_test_split(test_size=0.1, seed=42)
# For a fast, high-quality run, train on 2,000 samples (or increase to full dataset)
train_data = split["train"].select(range(min(2000, len(split["train"])) if "train" in split else range(2000)))
eval_data = split["test"].select(range(min(200, len(split["test"])) if "test" in split else range(200)))

def prepare(item):
    img = item["image"].convert("RGB") if isinstance(item["image"], Image.Image) else item["image"]
    caption = item["caption"][0] if isinstance(item["caption"], list) else str(item["caption"])
    encoded = processor(images=img, text=caption, padding="max_length", truncation=True, max_length=64)
    return {
        "pixel_values": encoded["pixel_values"][0],
        "input_ids": encoded["input_ids"][0],
        "attention_mask": encoded["attention_mask"][0],
        "labels": encoded["input_ids"][0],
    }

encoded_train = train_data.map(prepare, remove_columns=train_data.column_names, desc="Tokenizing Train")
encoded_eval = eval_data.map(prepare, remove_columns=eval_data.column_names, desc="Tokenizing Eval")
encoded_train.set_format("torch")
encoded_eval.set_format("torch")

In [ ]:
# 4. Training Arguments
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    num_train_epochs=3,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded_train,
    eval_dataset=encoded_eval,
)

print("Starting GPU training...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Training complete! Model saved to:", OUTPUT_DIR)

In [ ]:
# 5. Compress and Download Model to Local Project
!zip -r blip-flickr30k.zip ./blip-flickr30k

from google.colab import files
files.download('blip-flickr30k.zip')
print("Downloaded! Extract this into 'backend/models/blip-flickr30k' in your project.")